In [15]:
!pip install boto3 pandas

# 1. Import bibliotecile necesare pentru conectarea la AWS S3 si procesarea datelor
# 2. Setez variabilele de configurare pentru accesul la AWS S3

In [16]:
import boto3
import pandas as pd

from google.colab import userdata
AWS_ACCESS_KEY_ID = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION = "us-east-1"
BUCKET_NAME = "my-music-toplist-georgiana"
RAW_PREFIX = "raw/"

# 3. Conectez sesiunea boto3 folosind credentialele IAM

In [17]:
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

s3 = session.resource('s3')
bucket = s3.Bucket(BUCKET_NAME)

# 4. Listez toate fisierele din folderul /raw pentru a verifica daca sunt incarcate corect

In [18]:
for obj in bucket.objects.filter(Prefix=RAW_PREFIX):
  print(obj.key)

raw/
raw/deezer_top_20_2025-06-01.csv
raw/deezer_top_20_2025-06-02.csv
raw/deezer_top_20_2025-06-03.csv
raw/deezer_top_20_2025-06-04.csv
raw/deezer_top_20_2025-06-05.csv
raw/deezer_top_20_2025-06-06.csv
raw/deezer_top_20_2025-06-07.csv
raw/deezer_top_20_2025-06-08.csv
raw/deezer_top_20_2025-06-09.csv
raw/deezer_top_20_2025-06-10.csv
raw/deezer_top_20_2025-06-11.csv
raw/deezer_top_20_2025-06-12.csv
raw/deezer_top_20_2025-06-13.csv
raw/deezer_top_20_2025-06-14.csv
raw/deezer_top_20_2025-06-15.csv
raw/deezer_top_20_2025-06-16.csv
raw/deezer_top_20_2025-06-17.csv
raw/deezer_top_20_2025-06-18.csv
raw/deezer_top_20_2025-06-19.csv
raw/deezer_top_20_2025-06-20.csv
raw/deezer_top_20_2025-06-21.csv
raw/deezer_top_20_2025-06-22.csv
raw/deezer_top_20_2025-06-23.csv
raw/deezer_top_20_2025-06-24.csv
raw/deezer_top_20_2025-06-25.csv
raw/deezer_top_20_2025-06-26.csv
raw/deezer_top_20_2025-06-27.csv
raw/deezer_top_20_2025-06-28.csv
raw/deezer_top_20_2025-06-29.csv
raw/deezer_top_20_2025-06-30.csv
raw/i

# 5. Folosesc io.BytesIO pentru a citi fisierele direct din memorie, evitand descarcarea lor locala si eficientizand procesarea in cloud.
# 6. Combin toate fisierele intr-un singur DataFrame pentru analiza

In [19]:
import io

dfs = []

for obj in bucket.objects.filter(Prefix=RAW_PREFIX):
  if obj.key.endswith(".csv"):
    body = obj.get()['Body'].read()
    df = pd.read_csv(io.BytesIO(body))
    df["source_file"] = obj.key.split("/")[-1]
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)
full_df.head()

,position,artist,track,album,duration,date,source_file
0,1,Clairo,Juna,Charm,195,2025-06-01,deezer_top_20_2025-06-01.csv
1,2,Shaboozey,Good News,Good News,199,2025-06-01,deezer_top_20_2025-06-01.csv
2,3,Doechii,Anxiety,Anxiety,249,2025-06-01,deezer_top_20_2025-06-01.csv
3,4,Benson Boone,Beautiful Things,Beautiful Things,180,2025-06-01,deezer_top_20_2025-06-01.csv
4,5,Morgan Wallen,What I Want,I’m The Problem,184,2025-06-01,deezer_top_20_2025-06-01.csv


# 7. Extrag platforma (Spotify, Deezer, iTunes) din numele fisierului si o adaug ca o coloana noua

In [20]:
def extract_platform(filename):
  f = filename.lower()
  if "spotify" in f:
    return "Spotify"
  if "deezer" in f:
    return "Deezer"
    if "itunes" in f or "apple" in f:
      return "iTunes"
    return "Unknown"

full_df["platform"] = full_df["source_file"].apply(extract_platform)
full_df.head()

,position,artist,track,album,duration,date,source_file,platform
0,1,Clairo,Juna,Charm,195,2025-06-01,deezer_top_20_2025-06-01.csv,Deezer
1,2,Shaboozey,Good News,Good News,199,2025-06-01,deezer_top_20_2025-06-01.csv,Deezer
2,3,Doechii,Anxiety,Anxiety,249,2025-06-01,deezer_top_20_2025-06-01.csv,Deezer
3,4,Benson Boone,Beautiful Things,Beautiful Things,180,2025-06-01,deezer_top_20_2025-06-01.csv,Deezer
4,5,Morgan Wallen,What I Want,I’m The Problem,184,2025-06-01,deezer_top_20_2025-06-01.csv,Deezer


In [21]:
grouped = full_df.groupby("artist").agg(
    appearances=("artist", "count"),
    unique_tracks=("track", "nunique"),
    avg_position=("position", "mean"),
    best_position=("position", "min"),
    platforms=("platform", "nunique")
).reset_index()

# 8. Calculez "top 20 de artisti" in functie de numarul total de aparitii

In [22]:
top20 = grouped.sort_values(
    by=["platforms", "appearances", "avg_position"],
    ascending=[False, False, True]
).head(20)

top20

,artist,appearances,unique_tracks,avg_position,best_position,platforms
69,Sabrina Carpenter,91,4,12.318681,1,2
75,Teddy Swims,77,3,10.025974,1,2
9,Benson Boone,75,3,11.453333,1,2
42,Lady Gaga,59,4,11.966102,1,2
1,Alex Warren,56,1,10.964286,1,2
12,Billie Eilish,55,2,9.581818,1,2
58,Morgan Wallen,47,4,9.042553,1,2
15,Chappell Roan,44,2,8.386364,1,2
47,Lola Young,42,1,12.119048,1,2
16,Charli xcx,38,3,9.526316,1,2


# 9. Salvez rezultatul "top 20 artisti" intr-un fisier CSV local

In [23]:
output_filename = "top20_artists.csv"
top20.to_csv(output_filename, index=False)

# 10. Incarc fisierul CSV in folderul final/ din bucket-ul S3 pentru a-l putea accesa public

In [24]:
FINAL_PREFIX ="final/"

s3_client = session.client('s3')

s3_client.upload_file(
    Filename=output_filename,
    Bucket=BUCKET_NAME,
    Key=FINAL_PREFIX + output_filename
)